# CAPA

`CAPA` (Collective And Point Anomalies) jointly detects collective (segment) anomalies and point anomalies against a fixed baseline. Runtime is linear when there are many anomalies, which makes it the fast alternative to [CircularBinarySegmentation](circular_binseg.ipynb) whenever the baseline is well-defined and stable.

## Basic usage

The example below plants two short anomalous segments in an otherwise stationary series and recovers them with [L2Saving](../../api_reference/auto_generated/skchange.new_api.interval_scorers.L2Saving.rst) as the segment saving.

In [ ]:
import plotly.io as pio

from skchange.new_api.datasets import generate_piecewise_normal_data
from skchange.new_api.detectors import CAPA
from skchange.new_api.interval_scorers import L2Saving
from skchange.new_api.utils.plotting import plot_detections

pio.renderers.default = "notebook"

X = generate_piecewise_normal_data(
    means=[0, 5, 0, -4, 0],
    lengths=[40, 10, 60, 5, 35],
    seed=1,
)

detector = CAPA(L2Saving(), min_segment_length=3, penalty_scale=1.0)
detector.fit(X)
anomalies = detector.predict_segment_anomalies(X)

plot_detections(X, segment_anomalies=anomalies).show()
print(anomalies)

## Adding point anomalies
If you want the point anomalies detected by `CAPA` to be returned from `predict` alongside the segment anomalies, you can set `include_point_anomalies=True` and provide a `point_saving`. The `predict_all` method will return them separately if you need to distinguish between the two types of anomalies.

In [ ]:
detector = CAPA(
    segment_saving=L2Saving(),
    point_saving=L2Saving(),
    min_segment_length=3,
    include_point_anomalies=True,
)
result = detector.fit(X).predict_all(X)

print("Segment anomalies:", result["segment_anomalies"])
print("Point anomalies:  ", result["point_anomalies"])

## Sparse anomalies in multivariate data

The multivariate version of `CAPA` (MVCAPA) uses an *array-valued* penalty of length `n_features` with entry `i` interpreted as the penalty for `i+1` jointly affected features. At each candidate segment `CAPA` sorts the per-feature savings, picks the top `k` that maximises `sum(top_k_savings) - penalty[k-1]`, and reports both the segment and the identified feature subset. This makes `CAPA` well-suited to *sparse* anomalies where only a few features are affected.

The recommended penalty for MVCAPA is [mvcapa_penalty](../../api_reference/auto_generated/skchange.new_api.penalties.mvcapa_penalty.rst). The example below plants two anomalies affecting different feature subsets and reads off the identified features from `predict_all`.

In [ ]:
from skchange.new_api.penalties import mvcapa_penalty

X_mv = generate_piecewise_normal_data(
    means=[
        [0, 0, 0],
        [5, 0, 0],
        [0, 0, 0],
        [0, 4, -4],
        [0, 0, 0],
    ],
    lengths=[40, 10, 60, 8, 40],
    seed=0,
)
n_samples, n_features = X_mv.shape

detector = CAPA(
    L2Saving(),
    segment_penalty=mvcapa_penalty(n_samples, n_features),
    min_segment_length=3,
    penalty_scale=1.0,
)
result = detector.fit(X_mv).predict_all(X_mv)

for (start, end), features in zip(
    result["segment_anomalies"],
    result["segment_anomaly_features"],
):
    print(f"[{start:>3d}, {end:>3d})  affected features: {sorted(features.tolist())}")

plot_detections(
    X_mv,
    segment_anomalies=result["segment_anomalies"],
    affected_features=result["segment_anomaly_features"],
).show()

## Parameters worth knowing

- `segment_saving`: An interval scorer of type `saving` used for collective anomalies.
- `point_saving`: Optional interval scorer of type `saving` used for point anomalies. Required when `include_point_anomalies=True`.
- `segment_penalty`, `point_penalty`, `penalty_scale`: Thresholds applied to segment and point savings. Larger values produce fewer detections. Scalar or array; see [Penalties](../concepts/penalties.ipynb).
- `min_segment_length`, `max_segment_length`: Bounds on the length of segment anomalies.
- `include_point_anomalies`: Whether to jointly detect point anomalies.

See the full API reference for [CAPA](../../api_reference/auto_generated/skchange.new_api.detectors.CAPA.rst).

## See also

- [CircularBinarySegmentation](circular_binseg.ipynb): A more flexible transient-change detector when the baseline isn't fixed, at higher computational cost.
- [Change detectors](../concepts/change_detectors.ipynb): Background on the different search strategies.